# Bab 13 - Introducing Automatic Optimization: Building a Deep Learning Framework

**Nama:** Nanda Pratama ITM  
**NIM:** 101032330212  
**Kelas:** TK 47 01


> Notebook ini merupakan reproduksi edukatif dan pendalaman teori berdasarkan *Grokking Deep Learning* (Andrew W. Trask, Manning, 2019). Kode inti diadaptasi dari buku dan repositori pendamping resmi; perubahan kompatibilitas dijelaskan dalam dokumentasi.


## Capaian pembelajaran

Setelah menyelesaikan notebook ini, pembaca diharapkan mampu menjelaskan konsep utama bab, menghubungkannya dengan operasi NumPy, menjalankan contoh secara berurutan, dan menyebutkan batas interpretasinya.


## Tensor dan graf komputasi

Tensor menyimpan data serta hubungan operasi yang membentuknya. Graf komputasi merekam parent dan fungsi lokal sehingga gradien dapat dihitung otomatis dari output ke input.


## Autograd

Reverse-mode autodiff mengakumulasi vector-Jacobian product dari node keluaran. Jika tensor dipakai beberapa kali, semua kontribusi gradien harus dijumlahkan sebelum propagasi dilanjutkan.


## Layer, loss, optimizer

Layer mengemas parameter dan forward pass, loss mengubah prediksi-target menjadi skalar, dan optimizer memperbarui parameter dari gradien. Pemisahan tanggung jawab ini menjadi dasar framework modern.


## Embedding dan recurrence

Operasi indexing memerlukan scatter-add ketika gradien kembali ke tabel embedding. Layer berulang memakai parameter yang sama beberapa kali, sehingga akumulasi gradien dan urutan backward harus benar.


## Reproduksi kode dan eksperimen

Jalankan sel dari atas ke bawah. `MODE_CEPAT=True` menjaga waktu eksekusi singkat; ubah ke `False` untuk konfigurasi yang lebih panjang.


In [1]:
from pathlib import Path
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(1)
DATA_DIR = (Path.cwd().parent / "data").resolve()
MODE_CEPAT = True  # ubah ke False untuk eksperimen lebih panjang
assert DATA_DIR.is_dir(), f"Folder data tidak ditemukan: {DATA_DIR}"
print("Data:", DATA_DIR)
print("Mode cepat:", MODE_CEPAT)

def sigmoid(x):
    x = np.clip(x, -60, 60)
    return 1 / (1 + np.exp(-x))

def softmax(x):
    x = np.asarray(x)
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)


Data: D:\Kuliah\DL\UPLOAD_GITHUB\Grokking-Deep-Learning\data
Mode cepat: True


In [2]:
class Tensor:
    def __init__(self, data, creators=(), backward=None):
        self.data = np.asarray(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self.creators = creators
        self._backward = backward or (lambda: None)

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other))
        def backward():
            self.grad += np.broadcast_to(out.grad, self.data.shape)
            other.grad += np.broadcast_to(out.grad, other.data.shape)
        out._backward = backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other))
        def backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = backward
        return out

    def sum(self):
        out = Tensor(self.data.sum(), (self,))
        out._backward = lambda: self.grad.__iadd__(np.ones_like(self.data) * out.grad)
        return out

    def backward(self):
        order, visited = [], set()
        def build(node):
            if id(node) in visited: return
            visited.add(id(node))
            for parent in node.creators: build(parent)
            order.append(node)
        build(self)
        self.grad = np.ones_like(self.data)
        for node in reversed(order): node._backward()


In [3]:
x = Tensor([1, 2, 3])
w = Tensor([0.5, -0.2, 0.1])
loss = (x * w).sum()
loss.backward()
print("loss:", loss.data, "grad w:", w.grad, "expected:", x.data)
assert np.allclose(w.grad, x.data)


loss: 0.4 grad w: [1. 2. 3.] expected: [1. 2. 3.]


## Interpretasi dan kesimpulan

Hasil harus dibaca bersama asumsi, konfigurasi, seed, serta data yang digunakan. Keberhasilan menjalankan kode tidak membuktikan generalisasi, kausalitas, keamanan, atau kelayakan deployment. Bandingkan hasil dengan teori di atas dan jelaskan setiap perbedaan sebelum mengubah konfigurasi.
